In [6]:
from pydantic import BaseModel, Field, EmailStr, ValidationError
import re



class UserTest(BaseModel):
    id: int
    name: str

#Valid data
user1 = UserTest(id="1", name="Jonas" )
user1




UserTest(id=1, name='Jonas')

In [7]:
#Invalid data
user2 = UserTest(id = "Jonas", name= 2)

ValidationError: 2 validation errors for UserTest
id
  Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='Jonas', input_type=str]
    For further information visit https://errors.pydantic.dev/2.11/v/int_parsing
name
  Input should be a valid string [type=string_type, input_value=2, input_type=int]
    For further information visit https://errors.pydantic.dev/2.11/v/string_type

In [ ]:
class User(BaseModel):
    name: str = Field(min_length=2, max_length=30)
    age: int = Field(gt=-1, lt=126)
    email: EmailStr
    favourite_pet: str =Field(min_length=2, max_length=30)

In [ ]:
user3 = User(name="Jonas", age=40, email="jonas.gustafsson85@gmail.com", favourite_pet="dog")
user3

User(name='Jonas', age=40, email='jonas.gustafsson85@gmail.com', favourite_pet='dog')

In [ ]:
class User(BaseModel):
    name: str = Field(min_length=2, max_length=30)
    age: int = Field(gt=-1, lt=126)
    email: EmailStr
    favourite_pet: str =Field(min_length=2, max_length=30)

try:
    user4 = User(name="Jonas", age=40, email="jonas.gustafsson85@gmail.com", favourite_pet="dog")
except ValidationError as err:
    print(err)

In [ ]:
class Person_test:
    def __init__(self, name: str, age: int, email: str, favourite_pet: str):
        if not isinstance(name, str) or len(name) < 2 :
            raise ValueError("Name must be a a string with atleast 2 characters")
        self.name = name

        if not isinstance(age, int) or not(0 <= age < 126 ):
            raise ValueError("Age must be an integer between 0 and 125")
        self.age = age

        if not re.match(r"[^@]+@[^@]+\.[^@]+", email):
            raise ValueError("The email is not valid")
        self.email = email

        if not isinstance(favourite_pet, str) or len(favourite_pet) < 2 or len(favourite_pet) > 30:
            raise ValueError("The pet need to be atleast 2 characters and be a string")
        self.favourite_pet = favourite_pet



p1 = Person_test("Jonas", 40, "jonas@example.com", "dog")   # ✅ funkar
print("Valid:", vars(p1))

p2 = Person_test("J", -5, "not-an-email", "")              # ❌ kastar fel




Valid: {'name': 'Jonas', 'age': 40, 'email': 'jonas@example.com', 'favourite_pet': 'dog'}


ValueError: Name must be a a string with atleast 2 characters

# Validate data from API using python

In [11]:
import requests

headers = {"Accept": "application/json"}
response = requests.get("https://icanhazdadjoke.com/", headers=headers)

print(response.json())

{'id': 'ItWSnbUfVnb', 'joke': 'Why did the teddy bear say “no” to dessert? Because she was stuffed.', 'status': 200}


In [13]:
import requests

class Joke(BaseModel):
    id: int 
    joke: str

data = response.json()
joke_object = Joke(**data)

print(joke_object.joke)
print(joke_object.id)

ValidationError: 1 validation error for Joke
id
  Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='ItWSnbUfVnb', input_type=str]
    For further information visit https://errors.pydantic.dev/2.11/v/int_parsing

In [21]:
from pydantic import computed_field
class Joke(BaseModel):
    id: str 
    joke: str


    @computed_field
    @property
    def words_in_joke(self) -> int:
        """returns number of words in the joke"""
        return len(self.joke.split())

joke_obj = Joke(**data)
print("Skämt:", joke_obj.joke)
print("Antal ord:", joke_obj.words_in_joke)

Skämt: Why did the teddy bear say “no” to dessert? Because she was stuffed.
Antal ord: 13


In [38]:
import time
loop = 0
headers = {"Accept": "application/json"}
jokes_list4 = []

while(loop < 10):
    
    response = requests.get("https://icanhazdadjoke.com/", headers=headers)
    data = response.json()
    print(response.json())
    jokes_list4.append(data["joke"])
    loop = loop + 1
    time.sleep(2)

{'id': 'H6EY0Dljiyd', 'joke': 'I always wanted to look into why I procrastinate, but I keep putting it off. ', 'status': 200}
{'id': '2gaMZLBszsc', 'joke': 'Why did the kid throw the clock out the window? He wanted to see time fly!', 'status': 200}
{'id': 'EBAsPfiNuzd', 'joke': 'What did one plate say to the other plate? Dinner is on me!', 'status': 200}
{'id': '1DQZvXvX8Ed', 'joke': 'My first time using an elevator was an uplifting experience. The second time let me down.', 'status': 200}
{'id': 'nWDdFdFYLuc', 'joke': 'Have you heard about the film "Constipation", you probably haven\'t because it\'s not out yet.', 'status': 200}
{'id': 'It4Etrjiqrc', 'joke': 'Why does a chicken coop only have two doors? Because if it had four doors it would be a chicken sedan.', 'status': 200}
{'id': 'NCAIYLeNe', 'joke': 'I fear for the calendar, its days are numbered.\n', 'status': 200}
{'id': 'hNu4oORnOmb', 'joke': 'What do you call cheese by itself? Provolone.', 'status': 200}
{'id': '3gibFB5Muzd',

In [39]:
from pprint import pprint
pprint(jokes_list4)


['I always wanted to look into why I procrastinate, but I keep putting it '
 'off. ',
 'Why did the kid throw the clock out the window? He wanted to see time fly!',
 'What did one plate say to the other plate? Dinner is on me!',
 'My first time using an elevator was an uplifting experience. The second time '
 'let me down.',
 'Have you heard about the film "Constipation", you probably haven\'t because '
 "it's not out yet.",
 'Why does a chicken coop only have two doors? Because if it had four doors it '
 'would be a chicken sedan.',
 'I fear for the calendar, its days are numbered.\n',
 'What do you call cheese by itself? Provolone.',
 'Why was the robot angry? Because someone kept pressing his buttons!',
 'Why did the cookie cry?\r\nBecause his mother was a wafer so long']
